In [0]:
%load_ext autoreload
%autoreload 2

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from src.preprocess import correct_datetime, loan_type_preprocess
from src.feature_engineering import transaction_minute_extract
from src.validation import validate_data

In [0]:
catalog = 'finrisk360.default.'
df = spark.read.table(catalog+'brz_bank_transactions')
df.show(5)

In [0]:
df = correct_datetime(df,"transaction_date", "transaction_time")
df = loan_type_preprocess(df)
df.show(5)

In [0]:
drop_features = ['customer_id','transaction_id','transaction_datetime',
                 'account_balance','transaction_status',
                 'has_loan']

df = transaction_minute_extract(df, "transaction_datetime")
windowSpec = (
    Window.partitionBy("customer_id")
          .orderBy("transaction_datetime")
          .rowsBetween(Window.unboundedPreceding, -1)  # from start up to current row
)

df =  df.withColumn(
            "historical_average_amount",
            F.avg("transaction_amount").over(windowSpec))\
        .withColumn('prior_transaction_count',
                    F.count("transaction_amount").over(windowSpec)) \
        .withColumn(
            "amount_to_average_ratio",
            F.when(
                F.col("historical_average_amount").isNull() |
                (F.col("historical_average_amount") <= 0),
                F.lit(1.0))
            .otherwise(
                F.col("transaction_amount") /
                F.col("historical_average_amount"))) \
            .withColumn("amount_to_average_ratio",
                    F.round(F.col("amount_to_average_ratio"), 2)
            ).drop("historical_average_amount").fillna(0, ["prior_transaction_count"])

df = df.drop(*drop_features)
df.show(5)

In [0]:
validate_data(df)

In [0]:
df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(catalog+"slv_bank_transactions")

In [0]:
df = spark.read.table(catalog+'slv_bank_transactions')
df.show(5)

In [0]:
def stratified_train_val_test_split(
    df,
    label_col,
    train_frac=0.7,
    val_frac=0.15,
    test_frac=0.15,
    seed=42,
):
    if round(train_frac + val_frac + test_frac, 10) != 1.0:
        raise ValueError("train_frac + val_frac + test_frac must equal 1.0")

    window = Window.partitionBy(label_col).orderBy(F.rand(seed))
    counts = df.groupBy(label_col).count().withColumnRenamed("count", "label_count")

    split_df = (
        df.join(counts, on=label_col, how="left")
          .withColumn("row_num", F.row_number().over(window))
          .withColumn("train_cutoff", F.floor(F.col("label_count") * F.lit(train_frac)))
          .withColumn("val_cutoff", F.floor(F.col("label_count") * F.lit(train_frac + val_frac)))
    )

    train_df = split_df.filter(F.col("row_num") <= F.col("train_cutoff")).drop(
        "label_count", "row_num", "train_cutoff", "val_cutoff"
    )
    val_df = split_df.filter(
        (F.col("row_num") > F.col("train_cutoff")) &
        (F.col("row_num") <= F.col("val_cutoff"))
    ).drop("label_count", "row_num", "train_cutoff", "val_cutoff")
    test_df = split_df.filter(F.col("row_num") > F.col("val_cutoff")).drop(
        "label_count", "row_num", "train_cutoff", "val_cutoff"
    )

    return train_df, val_df, test_df


In [0]:
def summarize_stratified_splits(df, label_col="is_fraud"):
    train_df, val_df, test_df = stratified_train_val_test_split(df, label_col=label_col)

    for name, split_df in [("train", train_df), ("val", val_df), ("test", test_df)]:
        print(name, split_df.count(), split_df.groupBy(label_col).count().orderBy(label_col).collect())

    return train_df, val_df, test_df


train_df, val_df, test_df = summarize_stratified_splits(df)


In [0]:
validate_data(train_df)

In [0]:
validate_data(val_df)

In [0]:
validate_data(test_df)

In [0]:
final_catalog = 'gold_datasets.'
train_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(final_catalog+"train")
val_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(final_catalog+"val")
test_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(final_catalog+"test")